[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/14_kv_cache.ipynb)

# 🔴 Hard: KV Cache Attention

Implement **multi-head attention with KV caching** for efficient autoregressive generation.

During LLM inference, recomputing all key/value projections at every step is wasteful.
A **KV cache** stores previously computed K and V tensors so only the new token(s) need projection.

### Signature
```python
class KVCacheAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x: torch.Tensor, cache=None) -> tuple[torch.Tensor, tuple]:
        # x: (B, S_new, D) — new tokens
        # cache: None or (K_past, V_past) each (B, num_heads, S_past, d_k)
        # Returns: (output, (K_all, V_all))
```

### Requirements
- Inherit from `nn.Module`
- `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`: `nn.Linear` projections
- When `cache=None` (prefill): apply **causal mask**, return all K/V as cache
- When `cache` provided (decode): concat new K/V with cached, no causal mask needed for single-token decode
- Incremental decode must produce **identical** results to full forward pass

### Key Idea
```
Prefill:  [t0 t1 t2 t3] → full causal attention → cache = (K_{0:3}, V_{0:3})
Decode:   [t4]           → Q=t4, K/V=cache+t4  → cache = (K_{0:4}, V_{0:4})
Decode:   [t5]           → Q=t5, K/V=cache+t5  → cache = (K_{0:5}, V_{0:5})
```

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.1 MB/s eta 0:00:00


In [2]:

import torch
import torch.nn as nn
import math

In [23]:
# ✏️ YOUR IMPLEMENTATION HERE

class KVCacheAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.scale = (self.head_dim) ** -0.5
        pass  # Initialize W_q, W_k, W_v, W_o

    def forward(self, x, cache=None):
        q = torch.stack(self.W_q(x).chunk(self.num_heads,dim=-1), dim=1)
        k = torch.stack(self.W_k(x).chunk(self.num_heads,dim=-1), dim=1)
        v = torch.stack(self.W_v(x).chunk(self.num_heads,dim=-1), dim=1)
        if cache:
          k_cache, v_cache = cache[0], cache[1]
          k = torch.concat([k_cache, k], dim=2)
          v = torch.concat([v_cache, v], dim=2)
        logits = torch.einsum('bhov,bhiv->bhoi', q, k) * self.scale
        # mask = torch.triu(torch.ones(logits.shape[2], logits.shape[3]), diagonal=1).bool()
        mask = torch.triu(torch.ones(logits.shape[2], logits.shape[3]), diagonal=logits.shape[3]-logits.shape[2]+1).bool()
        logits = torch.masked_fill(logits, mask, float('-inf'))
        attn = torch.softmax(logits, dim=-1)
        output = torch.einsum('bhoi,bhiv->bhov', attn, v)
        output = output.transpose(1,2).reshape(output.shape[0], output.shape[2], output.shape[1] * output.shape[3])
        output = self.W_o(output)
        return (output, (k, v))
        # 1. Project Q, K, V from x
        # 2. Reshape to multi-head: (B, num_heads, S, d_k)
        # 3. If cache exists, concat new K/V with cached K/V
        # 4. Compute attention (causal mask needed during prefill)
        # 5. Return (output, (K_all, V_all))
        # pass

In [24]:
# 🧪 Debug
torch.manual_seed(0)
attn = KVCacheAttention(d_model=64, num_heads=4)
x = torch.randn(1, 6, 64)

# Full forward
full_out, _ = attn(x)
print("Full output shape:", full_out.shape)  # (1, 6, 64)

# Incremental: prefill 4, decode 1, decode 1
out1, cache = attn(x[:, :4])
print("Cache K shape:", cache[0].shape)  # (1, 4, 4, 16)
out2, cache = attn(x[:, 4:5], cache=cache)
out3, cache = attn(x[:, 5:6], cache=cache)
inc_out = torch.cat([out1, out2, out3], dim=1)
print("Match:", torch.allclose(full_out, inc_out, atol=1e-5))

Full output shape: torch.Size([1, 6, 64])
Cache K shape: torch.Size([1, 4, 4, 16])
Match: True


In [25]:
# ✅ SUBMIT
from torch_judge import check
check('kv_cache')


🧪 Testing: KV Cache Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (no cache) (8.7ms)
  ✅ [2/5] Cache structure (4.8ms)
  ✅ [3/5] Decode step appends to cache (5.0ms)
  ✅ [4/5] Incremental decode matches full forward (8.2ms)
  ✅ [5/5] Gradient flow (5.6ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (32.3ms total)
  Progress saved. Run status() to see your dashboard.

